# All metrics and significance tests

This notebook computes Atomic PropScore, Sentence PropScore, Sentence-BERT,
ROUGE-L, BERTScore, and BLEU from one annotated input dataset. Each metric is saved
to both JSON and a self-contained result notebook under `/content/`.

Expected input: `/content/merged_annotated_propositions.json`.

In [ ]:
!pip install -q transformers sentence-transformers torch numpy huggingface_hub nltk rouge-score bert-score pandas scipy tqdm nbformat

## Configuration and helpers

In [ ]:
import gc
import json
import math
from pathlib import Path

import nbformat as nbf
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

INPUT_FILE = Path('/content/merged_annotated_propositions.json')
OUTPUT_DIR = Path('/content')
GAMMAS = list(range(1, 31))
N_BOOTSTRAP = 1000
SEED = 42

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f'{INPUT_FILE} was not found. Upload merged_annotated_propositions.json '
        'to the Colab /content directory before running this notebook.'
    )

with INPUT_FILE.open('r', encoding='utf-8') as f:
    data = json.load(f)

required = {'query_id', 'reference_answer', 'generated_answer'}
for row_number, item in enumerate(data, start=1):
    missing = required.difference(item)
    if missing:
        raise KeyError(f'Row {row_number} is missing required keys: {sorted(missing)}')

print(f'Loaded {len(data)} records from {INPUT_FILE}')


def save_metric_results(metric_name, results, stem):
    """Save machine-readable JSON and a notebook containing the same results."""
    json_path = OUTPUT_DIR / f'{stem}.json'
    notebook_path = OUTPUT_DIR / f'{stem}.ipynb'

    with json_path.open('w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    result_nb = nbf.v4.new_notebook()
    result_nb['metadata']['kernelspec'] = {
        'display_name': 'Python 3', 'language': 'python', 'name': 'python3'
    }
    result_nb['cells'] = [
        nbf.v4.new_markdown_cell(
            f'# {metric_name} results\n\nComputed from `{INPUT_FILE}`.'
        ),
        nbf.v4.new_code_cell(
            "# The computed per-query results are embedded in this cell's output.\nresults",
            execution_count=1,
            outputs=[nbf.v4.new_output(
                output_type='display_data',
                data={'application/json': results, 'text/plain': f'{len(results)} scored records'},
                metadata={},
            )],
        ),
    ]
    nbf.write(result_nb, notebook_path)
    print(f'Saved {metric_name}: {json_path} and {notebook_path}')
    return json_path, notebook_path


def clean_text(value):
    if value is None:
        return ''
    return value if isinstance(value, str) else str(value)

## Atomic PropScore

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

embedder = SentenceTransformer('BAAI/bge-large-en-v1.5', device=str(device))
nli_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-large')
nli_model = AutoModelForSequenceClassification.from_pretrained(
    'cross-encoder/nli-deberta-v3-large'
).to(device).eval()

label2id = {str(k).lower(): int(v) for k, v in nli_model.config.label2id.items()}
ENTAILMENT_IDX = next(
    (idx for label, idx in label2id.items() if 'entail' in label),
    1,
)
print(f'NLI entailment label index: {ENTAILMENT_IDX}')


def get_nli_probabilities(pairs, batch_size=32):
    if not pairs:
        return []
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            encoded = nli_tokenizer(
                batch, padding=True, truncation=True, return_tensors='pt'
            ).to(device)
            probs = torch.softmax(nli_model(**encoded).logits, dim=1)
            probabilities.extend(probs[:, ENTAILMENT_IDX].cpu().tolist())
    return probabilities


def compute_prop_scores(items, reference_key, generated_key, detail_key):
    results = []
    for item in tqdm(items, desc=f'Computing {detail_key}', unit='query'):
        references = item.get(reference_key, []) or []
        candidates = item.get(generated_key, []) or []
        n_ref, n_cand = len(references), len(candidates)
        top_p = max(1, int(math.sqrt(n_cand))) if n_cand else 0

        if not references or not candidates:
            results.append({
                'query_id': item.get('query_id'),
                'question': item.get('question', ''),
                'prop-score': [{str(g): 0.0} for g in GAMMAS],
                detail_key: [{r: [{str(g): 0.0} for g in GAMMAS]} for r in references],
            })
            continue

        ref_embeddings = embedder.encode(
            references, normalize_embeddings=True, convert_to_tensor=True
        )
        cand_embeddings = embedder.encode(
            candidates, normalize_embeddings=True, convert_to_tensor=True
        )
        adjusted_similarity = torch.clamp(
            ref_embeddings @ cand_embeddings.T, min=0
        ).cpu().numpy()

        forward = [(r, c) for r in references for c in candidates]
        backward = [(c, r) for r in references for c in candidates]
        forward_probs = np.asarray(get_nli_probabilities(forward)).reshape(n_ref, n_cand)
        backward_probs = np.asarray(get_nli_probabilities(backward)).reshape(n_ref, n_cand)
        symmetric_probs = np.minimum(forward_probs, backward_probs)

        by_gamma = {gamma: [] for gamma in GAMMAS}
        details = []
        for ref_index, reference in enumerate(references):
            reference_scores = []
            for gamma in GAMMAS:
                base = adjusted_similarity[ref_index] * (1 + symmetric_probs[ref_index]) / 2
                exponent = 1 + gamma * (1 - symmetric_probs[ref_index])
                pair_scores = np.power(np.maximum(0, base), exponent)
                ranked = np.sort(pair_scores)[::-1][:top_p]
                aggregate = float(sum(
                    value / math.log2(rank + 2) for rank, value in enumerate(ranked)
                ))
                reference_scores.append({str(gamma): aggregate})
                by_gamma[gamma].append(aggregate)
            details.append({reference: reference_scores})

        results.append({
            'query_id': item.get('query_id'),
            'question': item.get('question', ''),
            'prop-score': [
                {str(g): float(np.mean(by_gamma[g]))} for g in GAMMAS
            ],
            detail_key: details,
        })
    return results


atomic_results = compute_prop_scores(
    data,
    'reference_answer_propositions',
    'generated_answer_propositions',
    'reference_answer_propositions',
)
save_metric_results('Atomic PropScore', atomic_results, 'atomic_propscore_results')

## Sentence PropScore

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

sentence_items = []
for item in data:
    enriched = dict(item)
    ref_text = clean_text(item.get('reference_answer'))
    gen_text = clean_text(item.get('generated_answer'))
    enriched['_reference_sentences'] = sent_tokenize(ref_text) if ref_text.strip() else []
    enriched['_generated_sentences'] = sent_tokenize(gen_text) if gen_text.strip() else []
    sentence_items.append(enriched)

sentence_results = compute_prop_scores(
    sentence_items,
    '_reference_sentences',
    '_generated_sentences',
    'reference_answer_sentences',
)
save_metric_results('Sentence PropScore', sentence_results, 'sentence_propscore_results')

# Release the large PropScore models before loading the baseline models.
del embedder, nli_model, nli_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## ROUGE-L, BERTScore, and BLEU

In [ ]:
from bert_score import score as bertscore
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer

references = [clean_text(item.get('reference_answer')) for item in data]
candidates = [clean_text(item.get('generated_answer')) for item in data]

_, _, bert_f1 = bertscore(
    candidates,
    references,
    lang='en',
    verbose=True,
    device=str(device),
)

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smoothing = SmoothingFunction().method1
rouge_results, bert_results, bleu_results = [], [], []

for index, item in enumerate(tqdm(data, desc='Computing text metrics')):
    reference = references[index]
    candidate = candidates[index]
    common = {'query_id': item.get('query_id'), 'question': item.get('question', '')}

    rouge_results.append({
        **common,
        'rouge-l': rouge.score(reference, candidate)['rougeL'].fmeasure,
    })
    bert_results.append({**common, 'bert-score': float(bert_f1[index].item())})

    try:
        ref_tokens = nltk.word_tokenize(reference)
        cand_tokens = nltk.word_tokenize(candidate)
    except LookupError:
        ref_tokens, cand_tokens = reference.split(), candidate.split()
    bleu_value = 0.0 if not ref_tokens or not cand_tokens else sentence_bleu(
        [ref_tokens], cand_tokens, smoothing_function=smoothing
    )
    bleu_results.append({**common, 'bleu': round(float(bleu_value), 6)})

save_metric_results('ROUGE-L', rouge_results, 'rouge_l_results')
save_metric_results('BERTScore', bert_results, 'bertscore_results')
save_metric_results('BLEU', bleu_results, 'bleu_results')

del bert_f1
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Sentence-BERT

In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=str(device))
ref_embeddings = sbert_model.encode(
    references, convert_to_tensor=True, show_progress_bar=True
)
cand_embeddings = sbert_model.encode(
    candidates, convert_to_tensor=True, show_progress_bar=True
)
cosine_scores = torch.nn.functional.cosine_similarity(ref_embeddings, cand_embeddings)

sbert_results = [
    {
        'query_id': item.get('query_id'),
        'question': item.get('question', ''),
        'sentence-bert-score': float(cosine_scores[index].item()),
    }
    for index, item in enumerate(data)
]
save_metric_results('Sentence-BERT', sbert_results, 'sentence_bert_results')

del sbert_model, ref_embeddings, cand_embeddings, cosine_scores
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Correlations with human scores

In [ ]:
master = {
    item['query_id']: {'human_score': item['human_score']}
    for item in data
    if 'human_score' in item
}


def add_scalar(results, source_key, target_key):
    for item in results:
        qid = item['query_id']
        if qid in master:
            master[qid][target_key] = item[source_key]


def add_gamma_scores(results, prefix):
    for item in results:
        qid = item['query_id']
        if qid in master:
            for score_dict in item.get('prop-score', []):
                for gamma, value in score_dict.items():
                    master[qid][f'{prefix}_gamma_{gamma}'] = value


add_scalar(rouge_results, 'rouge-l', 'rouge-l')
add_scalar(bert_results, 'bert-score', 'bert-score')
add_scalar(bleu_results, 'bleu', 'bleu')
add_scalar(sbert_results, 'sentence-bert-score', 'sentence-bert')
add_gamma_scores(atomic_results, 'atomic')
add_gamma_scores(sentence_results, 'sentence')

scores_df = pd.DataFrame.from_dict(master, orient='index').dropna()
if scores_df.empty:
    raise ValueError('No complete rows with human_score were available for evaluation.')

correlation_results = pd.DataFrame({
    'Pearson': scores_df.corr(method='pearson')['human_score'].drop('human_score'),
    'Kendall Tau': scores_df.corr(method='kendall')['human_score'].drop('human_score'),
}).sort_values('Kendall Tau', ascending=False)

display(correlation_results.round(4))
correlation_results.to_csv('/content/metric_correlations.csv', index_label='metric')

## Bootstrap significance tests

In [ ]:
from scipy.stats import kendalltau, pearsonr

baseline_metrics = ['bert-score', 'sentence-bert', 'rouge-l', 'bleu']
candidate_metrics = (
    correlation_results.loc[
        correlation_results.index.str.startswith(('atomic_gamma_', 'sentence_gamma_')),
        'Kendall Tau',
    ]
    .sort_values(ascending=False)
    .head(4)
    .index.tolist()
)


def bootstrap_comparison(proposed, baseline, method, n_iter=N_BOOTSTRAP):
    rng = np.random.default_rng(SEED)
    baseline_wins = 0
    valid = 0
    n_rows = len(scores_df)

    for _ in range(n_iter):
        positions = rng.integers(0, n_rows, n_rows)
        human = scores_df['human_score'].iloc[positions]
        proposed_values = scores_df[proposed].iloc[positions]
        baseline_values = scores_df[baseline].iloc[positions]

        if method == 'pearson':
            if min(np.var(human), np.var(proposed_values), np.var(baseline_values)) == 0:
                continue
            proposed_corr = pearsonr(human, proposed_values).statistic
            baseline_corr = pearsonr(human, baseline_values).statistic
        else:
            proposed_corr = kendalltau(human, proposed_values).statistic
            baseline_corr = kendalltau(human, baseline_values).statistic

        if np.isnan(proposed_corr) or np.isnan(baseline_corr):
            continue
        baseline_wins += baseline_corr >= proposed_corr
        valid += 1

    return baseline_wins / valid if valid else np.nan


significance_rows = []
for method, corr_column in [('pearson', 'Pearson'), ('kendall', 'Kendall Tau')]:
    for proposed in candidate_metrics:
        for baseline in baseline_metrics:
            probability = bootstrap_comparison(proposed, baseline, method)
            if probability < 0.05:
                result = 'Proposed significantly better'
            elif probability > 0.95:
                result = 'Baseline significantly better'
            else:
                result = 'No significant difference'
            significance_rows.append({
                'method': method,
                'proposed_metric': proposed,
                'proposed_correlation': correlation_results.loc[proposed, corr_column],
                'baseline_metric': baseline,
                'baseline_correlation': correlation_results.loc[baseline, corr_column],
                'p_baseline_ge_proposed': probability,
                'result': result,
            })

significance_df = pd.DataFrame(significance_rows)
display(significance_df.round(4))
significance_df.to_csv('/content/metric_significance_tests.csv', index=False)
print('Saved /content/metric_correlations.csv and /content/metric_significance_tests.csv')